# Leave-one-room-out test of the P13 network

Trains the P13 architecture from scratch with whole rooms or single recordings held out (`temporal_model/loro.py`).
Runtime: GPU (Runtime -> Change runtime type -> T4 GPU). About 20 minutes in total, most of it the 12 training runs.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


Clone the code (public repository, `main` branch), or update an existing clone.

In [2]:
import os, subprocess
if not os.path.exists('/content/Huawei'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Agata872/Huawei.git', '/content/Huawei'], check=True)
%cd /content/Huawei
!git pull -q
!git log --oneline -1
!pip install -q 'blosc2>=3,<5'

/content/Huawei
cdc7686 (grafted, HEAD -> main, origin/main, origin/HEAD) Add P13 multi-seed retraining, ensemble export, per-room calibration and leaderboard probes


Download the labeled archive (2.1 GB) from the link in `FutureNetwork-starting-kit/data_downloads.md`, check its SHA-256 and extract it.

In [3]:
%%bash
set -e
mkdir -p data/downloads data/extracted
cd data/downloads
[ -f FutureNetwork-data-labeled.zip ] || wget -q https://dl.r0h.eu/techarena2026-paris/FutureNetwork-data-labeled.zip
echo '1f60df2d4c1459c307e444741056e17a5ec537614cf577331ed04eb94d0765c1  FutureNetwork-data-labeled.zip' | sha256sum -c
unzip -q -n FutureNetwork-data-labeled.zip -d ../extracted
ls ../extracted/labeled | wc -l

FutureNetwork-data-labeled.zip: OK
97


Precompute the block maps (CPU, a few minutes) and check them against the raw-input features: the printed difference should be ~1e-6 or smaller.

In [4]:
!python -m temporal_model.maps --check

[1/48] a_a1_hash813694_1
[2/48] a_a1_hash058655_2
[3/48] a_a1_hash240174_2
[4/48] a_a1_hash861722_3
[5/48] a_a1_hash033659_3
[6/48] a_a1_hash844151_4
[7/48] a_a1_hash330776_4
[8/48] a_a1_hash420651_4
[9/48] a_a1_hash280746_3
[10/48] a_a1_hash069403_2
[11/48] a_a1_hash221231_1
[12/48] a_a1_hash957492_0
[13/48] c_c1_hash906339_1
[14/48] c_c1_hash072792_2
[15/48] c_c1_hash810891_3
[16/48] c_c1_hash292683_4
[17/48] c_c1_hash368203_4
[18/48] c_c1_hash672642_4
[19/48] c_c1_hash534150_4
[20/48] c_c1_hash419093_3
[21/48] c_c1_hash712526_3
[22/48] c_c1_hash884642_2
[23/48] c_c1_hash562262_2
[24/48] c_c1_hash347235_1
[25/48] b_b1_hash962037_0
[26/48] e_e1_hash508136_0
[27/48] e_e1_hash657193_0
[28/48] e_e1_hash250867_1
[29/48] unknown_unknown_hash287936_1
[30/48] unknown_unknown_hash461239_1
[31/48] unknown_unknown_hash081247_1
[32/48] g_g1_hash748206_0
[33/48] g_g1_hash299607_1
[34/48] g_g1_hash245884_1
[35/48] g_g1_hash284913_1
[36/48] g_g1_hash352161_0
[37/48] g_g1_hash335239_1
[38/48] g_g1_h

Train and score: c1 and a1 held out as whole rooms, and one recording per headcount held out inside each room, three seeds each.

In [5]:
!python -m temporal_model.loro --holdout room:c1,room:a1,traces:c1,traces:a1 --seeds 0,1,2

room:c1 seed 0: train 36 traces, test ['c_c1_hash906339_1', 'c_c1_hash072792_2', 'c_c1_hash810891_3', 'c_c1_hash292683_4', 'c_c1_hash368203_4', 'c_c1_hash672642_4', 'c_c1_hash534150_4', 'c_c1_hash419093_3', 'c_c1_hash712526_3', 'c_c1_hash884642_2', 'c_c1_hash562262_2', 'c_c1_hash347235_1']
    step 500: loss 0.388 (15 s)
    step 1000: loss 0.230 (28 s)
    step 1500: loss 0.227 (41 s)
    step 2000: loss 0.228 (54 s)
    step 2500: loss 0.232 (67 s)
    step 3000: loss 0.228 (79 s)
  balanced 0.225 (chance 0.25), Spearman window -0.24 trace -0.35
    true 1: [7, 32, 1, 0, 0]
    true 2: [28, 26, 6, 0, 0]
    true 3: [17, 18, 25, 0, 0]
    true 4: [37, 41, 2, 0, 0]
room:c1 seed 1: train 36 traces, test ['c_c1_hash906339_1', 'c_c1_hash072792_2', 'c_c1_hash810891_3', 'c_c1_hash292683_4', 'c_c1_hash368203_4', 'c_c1_hash672642_4', 'c_c1_hash534150_4', 'c_c1_hash419093_3', 'c_c1_hash712526_3', 'c_c1_hash884642_2', 'c_c1_hash562262_2', 'c_c1_hash347235_1']
    step 500: loss 0.730 (12 s)
   

Zip the per-window results (logits included) for the local repository's `runs/temporal_model/`. The browser download only works in the Colab web UI; elsewhere fetch `/content/loro_results.zip` from the runtime's file browser.

In [6]:
!cd runs && zip -qr /content/loro_results.zip temporal_model
try:
    from google.colab import files
    files.download('/content/loro_results.zip')
except Exception as error:
    print(f'browser download unavailable ({error}); the zip is at /content/loro_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>